In [0]:
%python
# Set up Auto Loader stream to ingest CSV files
from pyspark.sql.functions import *

# Define paths
source_path = "/Volumes/dev/autoloader/raw/sales/"
schema_location = "/Volumes/dev/autoloader/raw/autoloader_schemaLocation/"
checkpoint_path = "/Volumes/dev/autoloader/raw/autoloader_checkpoint/"
target_table = "dev.autoloader.sales_autoloader"

# Read stream using Auto Loader
df_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("header", "true")
    .option("inferSchema", "true")
    .load(source_path)
)

# Write stream to Delta table
query = (df_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

